In [ ]:
#@title Загрузка, Декомпиляция и Сохранение { display-mode: "form" }
import os
import requests
import zipfile
import json
import re
import sys
import shutil
import logging
from google.colab import files
from tqdm.notebook import tqdm

URL = "https://originn0.github.io/dynamic_social_democracy/core.js" #@param {type:"string"}
OUTPUT_DIR = 'decompiled_output'
ZIP_OUTPUT_PATH = 'source_backup.zip'
CORE_JS_PATH = 'core.js'

logging.basicConfig(level=logging.INFO, format='%(message)s')
logger = logging.getLogger("decompiler")

def magic_to_logic(magic, root_type='predicate'):
    if not magic: return ""
    res = magic.strip()
    if "\n" in res or "//" in res or "/*" in res or "var " in res or "let " in res or "const " in res or "if (" in res or "{" in res:
        return f"{{!\n{res}\n!}}"
    if res.startswith("return "): res = res[7:]
    if res.endswith(";"): res = res[:-1]
    res = re.sub(r"Q\[['\"](.*?)['\"]\]", r"\1", res)
    res = re.sub(r"Q\.([a-zA-Z0-9_]+)", r"\1", res)
    res = re.sub(r"state\.visits\[['\"](.*?)['\"]\]", r"@\1", res)
    res = re.sub(r"\(?([\w@\.]+)\s*\|\|\s*0\)?", r"\1", res)
    if root_type in ['predicate', 'expression']:
        res = res.replace("===", "=").replace("==", "=").replace("!==", "!=").replace("!=", "!=")
        res = res.replace(" && ", " and ").replace(" || ", " or ")
        while True:
            new_res = re.sub(r"\(([\w@\.\-]+)\)", r"\1", res)
            if new_res == res: break
            res = new_res
    if root_type == 'actions':
        res = re.sub(r"(\w+)\s*=\s*\1\s*([\+\-\*\/])\s*(.*?)$", r"\1 \2= \3", res)
        if "=" not in res: res = re.sub(r"(\w+)\s*=\s*(.*?)$", r"\1 = \2", res)
    return res.strip()

def reconstruct_content(content_obj, state_deps, is_one_line=False):
    if content_obj is None: return ""
    if isinstance(content_obj, str): return content_obj
    if isinstance(content_obj, list):
        parts = [reconstruct_content(item, state_deps, is_one_line) for item in content_obj]
        return "".join(parts)
    if isinstance(content_obj, dict):
        ctype = content_obj.get("type")
        inner = content_obj.get("content", "")
        if ctype == "paragraph": return reconstruct_content(inner, state_deps, is_one_line) + ("" if is_one_line else "\n\n")
        if ctype == "heading": return "= " + reconstruct_content(inner, state_deps, is_one_line).strip() + ("" if is_one_line else "\n\n")
        if ctype == "emphasis-1": return "*" + reconstruct_content(inner, state_deps, True) + "*"
        if ctype == "emphasis-2": return "**" + reconstruct_content(inner, state_deps, True) + "**"
        if ctype == "emphasis-3": return "***" + reconstruct_content(inner, state_deps, True) + "***"
        if ctype == "line-break": return "\n"
        if ctype == "blockquote": return "> " + reconstruct_content(inner, state_deps, is_one_line)
        if ctype == "hrule": return "---"
        if ctype == "conditional":
            idx = content_obj.get('predicate')
            logic = "UNKNOWN"
            if state_deps and idx < len(state_deps): logic = magic_to_logic(state_deps[idx].get("fn", {}).get("$code", ""), "predicate")
            return f"[? if {logic} : {reconstruct_content(inner, state_deps, True).strip()} ?]"
        if ctype == "insert":
            idx = content_obj.get('insert')
            logic = "UNKNOWN"
            if state_deps and idx < len(state_deps): logic = magic_to_logic(state_deps[idx].get("fn", {}).get("$code", ""), "expression")
            return f"[? {logic} ?]"
    return str(content_obj)

SCENE_PROPS = [
    ("title", "title"), ("subtitle", "subtitle"), ("unavailableSubtitle", "unavailable-subtitle"),
    ("viewIf", "view-if"), ("chooseIf", "choose-if"), ("onArrival", "on-arrival"), ("onDeparture", "on-departure"), 
    ("onDisplay", "on-display"), ("maxVisits", "max-visits"), ("countVisitsMax", "count-visits-max"), 
    ("maxVisitsVar", "max-visits-var"), ("priority", "priority"), ("tags", "tags"), ("newPage", "new-page"),
    ("setRoot", "set-root"), ("isSpecial", "is-special"), ("gameOver", "game-over"),
    ("goTo", "go-to"), ("goSub", "go-sub"), ("setJump", "set-jump"), ("call", "call"),
    ("setBg", "set-bg"), ("audio", "audio"), ("faceImage", "face-image"),
    ("cardImage", "card-image"), ("wideImage", "wide-image"), ("bannerImage", "banner-image"),
    ("isDeck", "is-deck"), ("isCard", "is-card"), ("isHand", "is-hand"), ("isPinnedCard", "is-pinned-card"),
    ("maxCards", "max-cards"), ("checkQuality", "check-quality"), ("broadDifficulty", "broad-difficulty"),
    ("narrowDifficulty", "narrow-difficulty"), ("difficultyScaler", "difficulty-scaler"),
    ("difficultyIncrement", "difficulty-increment"), ("checkSuccessGoTo", "check-success-go-to"),
    ("checkFailureGoTo", "check-failure-go-to"), ("minChoices", "min-choices"), ("maxChoices", "max-choices"),
    ("isTop", "is-top"), ("setSprites", "set-sprites"), ("setSpriteStyles", "set-sprite-styles"),
    ("setTopLeftStyle", "set-top-left-style"), ("setTopRightStyle", "set-top-right-style"),
    ("setBottomLeftStyle", "set-bottom-left-style"), ("setBottomRightStyle", "set-bottom-right-style")
]

def decompile_scene_heuristic(scene, root_id):
    lines = []
    is_root = scene['id'] == root_id
    if not is_root:
        sid = scene['id']
        if sid.startswith(root_id + "."): sid = "@" + sid[len(root_id)+1:]
        lines.append(sid)
    handled = {"id", "content", "options", "$metadata", "type", "stateDependencies"}
    for js_p, dry_p in SCENE_PROPS:
        if js_p in scene:
            handled.add(js_p); val = scene[js_p]
            if val is None: continue
            if js_p == "countVisitsMax" and scene.get("maxVisits") == val: continue
            if js_p in ["subtitle", "unavailableSubtitle"] and isinstance(val, dict): lines.append(f"{dry_p}: {reconstruct_content(val.get('content'), val.get('stateDependencies'), True).strip()}")
            elif js_p in ["viewIf", "chooseIf", "maxVisitsVar"]: lines.append(f"{dry_p}: {magic_to_logic(val.get('$code') if isinstance(val, dict) else '', 'predicate')}")
            elif js_p in ["onArrival", "onDeparture", "onDisplay"]:
                actions = [magic_to_logic(act.get("$code", ""), "actions") for act in val]
                lines.append(f"{dry_p}: " + "\n".join(actions) if any("{!" in a for a in actions) else f"{dry_p}: {'; '.join(actions)}")
            elif js_p == "tags": lines.append(f"{dry_p}: {', '.join(val) if isinstance(val, list) else val}")
            elif js_p in ["goTo", "goSub"]:
                parts = [item["id"] + (" if " + magic_to_logic(item['predicate'].get('$code', ''), 'predicate') if "predicate" in item else "") for item in val]
                lines.append(f"{dry_p}: {'; '.join(parts)}")
            else: lines.append(f"{dry_p}: {str(val).lower() if isinstance(val, bool) else val}")
    for k, v in scene.items():
        if k not in handled and not k.startswith('$'): lines.append(f"{k}: {v}")
    cont = scene.get("content")
    if cont:
        text = reconstruct_content(cont.get("content"), cont.get("stateDependencies") or scene.get("stateDependencies")).strip() if isinstance(cont, dict) else str(cont).strip()
        if text: lines.append(""); lines.append(text)
    if "options" in scene:
        lines.append("")
        for opt in scene["options"]:
            oid, otit = opt['id'], opt.get('title', '')
            if oid.startswith(root_id + "."): oid = "@" + oid[len(root_id)+1:]
            line = f"- {oid}"
            if otit: 
                t = reconstruct_content(opt['title'].get('content') if isinstance(opt['title'], dict) else opt['title'], opt['title'].get('stateDependencies') if isinstance(opt['title'], dict) else None, True).strip()
                line += f": {t}"
            lines.append(line)
            for k, v in opt.items():
                if k in ['id', 'title']: continue
                dKey = k.replace('ViewIf', 'view-if').replace('ChooseIf', 'choose-if')
                lines.append(f"  {dKey}: {magic_to_logic(v['$code'], 'predicate') if isinstance(v, dict) and '$code' in v else v}")
    return "\n".join(lines)

def download_with_progress(url, filename):
    print(f"🌐 Подключение к {url}...")
    r = requests.get(url, stream=True); r.raise_for_status()
    total = int(r.headers.get('content-length', 0))
    with open(filename, 'wb') as f, tqdm(desc="⬇️ Скачивание", total=total, unit='iB', unit_scale=True) as bar:
        for chunk in r.iter_content(chunk_size=1024): bar.update(f.write(chunk))
    print(f"✅ Файл {filename} скачан!\n")

def zip_directory(folder_path, zip_path):
    print(f"📦 Упаковка результатов в {zip_path}...")
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for root, _, files in os.walk(folder_path):
            for file in tqdm(files, desc="🗜️ Архивирование"): zipf.write(os.path.join(root, file), os.path.relpath(os.path.join(root, file), folder_path))
    print(f"✅ Архив создан!\n")

def run_decompiler(core_path, out_dir):
    with open(core_path, 'r', encoding='utf-8') as f: content = f.read()
    prefix = "window.game="
    start_idx = content.find(prefix)
    json_part = content[start_idx + len(prefix):] if start_idx != -1 else content.strip()
    if json_part.endswith(';'): json_part = json_part[:-1]
    decoder = json.JSONDecoder()
    wrapper, _ = decoder.raw_decode(json_part)
    game = json.loads(wrapper['compiled']) if isinstance(wrapper, dict) and 'compiled' in wrapper else wrapper
    if os.path.exists(out_dir): shutil.rmtree(out_dir)
    os.makedirs(out_dir)
    
    # Metadata restoration
    files_metadata = {}
    def find_metadata(obj):
        if isinstance(obj, dict):
            if '$metadata' in obj and '$raw' in obj['$metadata']: files_metadata[obj['$metadata']['$file']] = obj['$metadata']['$raw']
            for v in obj.values(): find_metadata(v)
        elif isinstance(obj, list): [find_metadata(i) for i in obj]
    find_metadata(game)
    
    if files_metadata:
        print("✨ Найдена метаинформация. Восстановление 1 в 1.")
        for fp, raw in tqdm(files_metadata.items(), desc="Restoring"): 
            rel = fp.split('source/', 1)[1] if 'source/' in fp else os.path.basename(fp)
            out = os.path.join(out_dir, rel); os.makedirs(os.path.dirname(out), exist_ok=True)
            with open(out, 'w', encoding='utf-8') as f: f.write(raw)
        return
    
    print("⚠️ Метаинформация отсутствует. Запуск эвристической реконструкции.")
    with open(os.path.join(out_dir, "info.dry"), 'w', encoding='utf-8') as f: f.write(f"title: {game.get('title', '')}\nauthor: {game.get('author', '')}\nifid: {game.get('ifid', '')}\n")
    qds = game.get("qdisplays", {})
    if qds:
        os.makedirs(os.path.join(out_dir, "qdisplays"), exist_ok=True)
        for qid, qd in qds.items():
            lines = [""] + [f"({i.get('min', '')}..{i.get('max', '')}) {reconstruct_content(i.get('output'), None, True)}" for i in qd.get("content", [])]
            with open(os.path.join(out_dir, "qdisplays", f"{qid}.qdisplay.dry"), 'w', encoding='utf-8') as f: f.write("\n".join(lines))
    
    scenes = game.get("scenes", {})
    groups, internal_ids = {}, ['prevScene', 'prevTopScene', 'jumpScene', 'backSpecialScene', 'returnScene']
    for sid, scene in scenes.items():
        if sid in internal_ids: continue
        bid = sid.split('.')[0]
        if bid not in groups: groups[bid] = []
        groups[bid].append(scene)
    
    for bid, sub in tqdm(groups.items(), desc="Reconstructing"):
        sub.sort(key=lambda s: (s['id'] != bid, s['id']))
        blocks = [decompile_scene_heuristic(s, bid) for s in sub]
        out = os.path.join(out_dir, "scenes", f"{bid}.scene.dry"); os.makedirs(os.path.dirname(out), exist_ok=True)
        with open(out, 'w', encoding='utf-8') as f: f.write("\n\n".join(blocks) + "\n")

if URL.strip(): download_with_progress(URL, CORE_JS_PATH)
else:
    print("⚠️ URL не указан. Загрузите core.js вручную:")
    uploaded = files.upload()
    if uploaded: CORE_JS_PATH = list(uploaded.keys())[0]

if os.path.exists(CORE_JS_PATH):
    print(f"⚙️ Обработка {CORE_JS_PATH}...")
    run_decompiler(CORE_JS_PATH, OUTPUT_DIR)
    zip_directory(OUTPUT_DIR, ZIP_OUTPUT_PATH)
    files.download(ZIP_OUTPUT_PATH)
    print("✅ Готово!")
else: print("❌ Ошибка: core.js не найден.")